In [ ]:
import pandas as pd
df = pd.read_csv("CRMLSSold_with_rates.csv")
print(df.shape)  

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_24626/2175430653.py:2: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: ListAgentEmail, 3: FireplaceYN, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSSold_with_rates.csv")


(397603, 86)


Step 1  Process Date Fields

In [57]:

date_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

print(df[date_cols].dtypes)
print(df[date_cols].head(3))

CloseDate                   str
PurchaseContractDate        str
ListingContractDate         str
ContractStatusChangeDate    str
dtype: object
    CloseDate PurchaseContractDate ListingContractDate  \
0  2024-01-26           2023-11-22          2021-10-06   
1  2024-01-05           2021-06-30          2021-03-08   
2  2024-01-05           2021-11-18          2021-03-08   

  ContractStatusChangeDate  
0               2024-01-26  
1               2024-01-05  
2               2024-01-05  


In [58]:
# Convert the date column to datetime.
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
print(df[date_cols].dtypes)
print(df[date_cols].head(5))

CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object
   CloseDate PurchaseContractDate ListingContractDate ContractStatusChangeDate
0 2024-01-26           2023-11-22          2021-10-06               2024-01-26
1 2024-01-05           2021-06-30          2021-03-08               2024-01-05
2 2024-01-05           2021-11-18          2021-03-08               2024-01-05
3 2024-01-30           2024-08-05          2024-01-30               2024-01-30
4 2024-01-29           2024-01-29          2024-01-29               2024-01-29


In [59]:
# create date consistency flags

# A. listing date is after close date
df["listing_after_close_flag"] = (
    df["ListingContractDate"].notna() &
    df["CloseDate"].notna() &
    (df["ListingContractDate"] > df["CloseDate"])
)

# B. purchase date is after close date
df["purchase_after_close_flag"] = (
    df["PurchaseContractDate"].notna() &
    df["CloseDate"].notna() &
    (df["PurchaseContractDate"] > df["CloseDate"])
)

# C. negative timeline:
# listing date is after purchase date
df["negative_timeline_flag"] = (
    df["ListingContractDate"].notna() &
    df["PurchaseContractDate"].notna() &
    (df["ListingContractDate"] > df["PurchaseContractDate"])
)
#  convert True/False to 1/0
df["listing_after_close_flag"] = df["listing_after_close_flag"].astype(int)
df["purchase_after_close_flag"] = df["purchase_after_close_flag"].astype(int)
df["negative_timeline_flag"] = df["negative_timeline_flag"].astype(int)

#  check flag counts
print("Flag counts:")
print("listing_after_close_flag:", df["listing_after_close_flag"].sum())
print("purchase_after_close_flag:", df["purchase_after_close_flag"].sum())
print("negative_timeline_flag:", df["negative_timeline_flag"].sum())
print()

Flag counts:
listing_after_close_flag: 58
purchase_after_close_flag: 240
negative_timeline_flag: 261



In [60]:
print("Rows with listing_after_close_flag = 1")
print(df.loc[df["listing_after_close_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

print("\nRows with purchase_after_close_flag = 1")
print(df.loc[df["purchase_after_close_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

print("\nRows with negative_timeline_flag = 1")
print(df.loc[df["negative_timeline_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

Rows with listing_after_close_flag = 1
      ListingContractDate PurchaseContractDate  CloseDate
22             2024-01-31           2024-01-15 2024-01-30
83             2024-01-26           2024-01-10 2024-01-25
711            2024-01-02           2023-12-04 2024-01-01
24279          2024-04-08           2024-03-27 2024-03-29
24399          2024-03-21           2024-03-03 2024-03-20

Rows with purchase_after_close_flag = 1
   ListingContractDate PurchaseContractDate  CloseDate
3           2024-01-30           2024-08-05 2024-01-30
7           2024-01-31           2024-03-04 2024-01-31
11          2024-01-31           2024-02-05 2024-01-31
12          2024-01-29           2024-02-04 2024-01-29
20          2024-01-29           2024-02-01 2024-01-29

Rows with negative_timeline_flag = 1
    ListingContractDate PurchaseContractDate  CloseDate
22           2024-01-31           2024-01-15 2024-01-30
32           2024-01-18           2024-01-05 2024-01-30
48           2024-01-29           20

In [61]:
# High-Missing Columns (≥90%)
missing_pct = df.isnull().mean() * 100
high_missing_cols = missing_pct[missing_pct >= 90].index.tolist()
print("High-missing cols:", len(high_missing_cols))

# Non-analytical Column
extra_drop = [
    "BuyerAgentAOR", "ListAgentAOR",
    "ListAgentEmail",
    "ListAgentFirstName", "ListAgentLastName",
    "ListAgentFullName",
    "BuyerAgentFirstName", "BuyerAgentLastName",
    "CoListAgentFirstName", "CoListAgentLastName",
    "ListOfficeName", "BuyerOfficeName", "CoListOfficeName",
    "BuyerOfficeAOR", "BuyerAgencyCompensationType", "BuyerAgencyCompensation"
]
extra_drop = [col for col in extra_drop if col in df.columns]
print("Manual drop cols:", len(extra_drop))

before_cols = df.shape[1]
# Execute Deletion
df = df.drop(columns=high_missing_cols)
df = df.drop(columns=extra_drop)

after_cols = df.shape[1]

print("===== Check Deletion =====")
print("Before:", before_cols)
print("After:", after_cols)
print("Removed:", before_cols - after_cols)

High-missing cols: 17
Manual drop cols: 16
===== Check Deletion =====
Before: 89
After: 56
Removed: 33


In [62]:
# ---------- Duplicate Column Check ----------
# 1. Suffix duplicates (.1, .2, etc.)
suffix_dups = [col for col in df.columns if col.endswith((".1", ".2", ".3"))]

# 2. Duplicate column names
name_dups = df.columns[df.columns.duplicated()].tolist()

# 3. Duplicate content columns
content_dups = []
cols = df.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if df[cols[i]].equals(df[cols[j]]):
            content_dups.append((cols[i], cols[j]))

# ---------- Print Summary ----------
print("===== Duplicate Column Summary =====")
print(f"Suffix duplicates (.1 etc): {len(suffix_dups)}")
print(f"Duplicate column names: {len(name_dups)}")
print(f"Duplicate content pairs: {len(content_dups)}")

print("\n===== Duplicate Content Examples =====")

for col1, col2 in content_dups:
    print(f"\nComparing: {col1} vs {col2}")
    print(df[[col1, col2]].head(5))

===== Duplicate Column Summary =====
Suffix duplicates (.1 etc): 0
Duplicate column names: 0
Duplicate content pairs: 2

===== Duplicate Content Examples =====

Comparing: ListingKey vs ListingKeyNumeric
   ListingKey  ListingKeyNumeric
0   551985747          551985747
1   522107581          522107581
2   510919001          510919001
3  1079166779         1079166779
4  1075037759         1075037759

Comparing: latfilled vs lonfilled
  latfilled lonfilled
0       NaN       NaN
1       NaN       NaN
2       NaN       NaN
3       NaN       NaN
4       NaN       NaN


In [ ]:

# Drop true duplicate column
df = df.drop(columns=["ListingKeyNumeric"])
print("Dropped ListingKeyNumeric (duplicate of ListingKey)")

print(df[["latfilled", "lonfilled"]].isnull().mean())
#Although latfilled and lonfilled appeared identical in some cases, further
# inspection showed that they primarily consist of missing values and represent data 
# quality flags rather than true duplicate variables. Therefore, they were not removed
#  as duplicate columns.

Dropped ListingKeyNumeric (duplicate of ListingKey)
latfilled    0.839327
lonfilled    0.839327
dtype: float64


In [69]:
#Check to ensure that the data type is correct
num_cols = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger"
]

print(df[num_cols].dtypes)
for col in num_cols:
    print(f"\n--- {col} sample values ---")
    print(df[col].head(3))
for col in num_cols:
    print(f"{col} missing after conversion:", df[col].isna().sum())

ClosePrice               float64
ListPrice                float64
LivingArea               float64
DaysOnMarket               int64
BedroomsTotal            float64
BathroomsTotalInteger    float64
dtype: object

--- ClosePrice sample values ---
0    240000.0
1    815000.0
2    810000.0
Name: ClosePrice, dtype: float64

--- ListPrice sample values ---
0    295000.0
1    759900.0
2    770000.0
Name: ListPrice, dtype: float64

--- LivingArea sample values ---
0    1140.0
1    1974.0
2    1974.0
Name: LivingArea, dtype: float64

--- DaysOnMarket sample values ---
0    777
1     33
2    228
Name: DaysOnMarket, dtype: int64

--- BedroomsTotal sample values ---
0    2.0
1    4.0
2    4.0
Name: BedroomsTotal, dtype: float64

--- BathroomsTotalInteger sample values ---
0    2.0
1    4.0
2    4.0
Name: BathroomsTotalInteger, dtype: float64
ClosePrice missing after conversion: 2
ListPrice missing after conversion: 0
LivingArea missing after conversion: 229
DaysOnMarket missing after conversion: 

Step 3: Outlier Check (Invalid Values)

In [70]:

# Invalid value flags 
# -------------------------------

df["invalid_close_price_flag"] = (df["ClosePrice"] <= 0).astype(int)
df["invalid_living_area_flag"] = (df["LivingArea"] <= 0).astype(int)
df["invalid_days_on_market_flag"] = (df["DaysOnMarket"] < 0).astype(int)
df["invalid_bedrooms_flag"] = (df["BedroomsTotal"] < 0).astype(int)
df["invalid_bathrooms_flag"] = (df["BathroomsTotalInteger"] < 0).astype(int)

# overall flag
df["invalid_numeric_flag"] = df[
    [
        "invalid_close_price_flag",
        "invalid_living_area_flag",
        "invalid_days_on_market_flag",
        "invalid_bedrooms_flag",
        "invalid_bathrooms_flag"
    ]
].max(axis=1)

# count
print("Invalid Value Counts:")
print(df[
    [
        "invalid_close_price_flag",
        "invalid_living_area_flag",
        "invalid_days_on_market_flag",
        "invalid_bedrooms_flag",
        "invalid_bathrooms_flag",
        "invalid_numeric_flag"
    ]
].sum())

Invalid Value Counts:
invalid_close_price_flag         1
invalid_living_area_flag       144
invalid_days_on_market_flag     46
invalid_bedrooms_flag            0
invalid_bathrooms_flag           0
invalid_numeric_flag           191
dtype: int64


In [10]:
# preview rows with any invalid numeric issue
invalid_rows = df[df["invalid_numeric_flag"] == 1]

print("Rows with any invalid numeric issue:")
print(invalid_rows[
    [
        "ClosePrice",
        "LivingArea",
        "DaysOnMarket",
        "BedroomsTotal",
        "BathroomsTotalInteger"
    ]
].head(10))

Rows with any invalid numeric issue:
       ClosePrice  LivingArea  DaysOnMarket  BedroomsTotal  \
6       1950000.0      2100.0           -36            3.0   
7708    8000000.0         0.0            62            4.0   
11214    899000.0      2533.0           -13            2.0   
11381    372000.0      1488.0           -10            3.0   
18698    700000.0      1382.0           -10            4.0   
21099   9000000.0         0.0            76            4.0   
22968   1930000.0         0.0           128            4.0   
22969   1850000.0         0.0            94            4.0   
23005    476985.0      1342.0           -13            3.0   
24397    799000.0      1284.0           -48            3.0   

       BathroomsTotalInteger  
6                        0.0  
7708                     5.0  
11214                    3.0  
11381                    2.0  
18698                    2.0  
21099                    3.0  
22968                    4.0  
22969                    4.0  
2

Step 4：Geographic Data Checks

In [71]:
# -------------------------------
# Geographic flags
# -------------------------------

# 1️ Missing Values
df["missing_geo_flag"] = (
    df["Latitude"].isna() | df["Longitude"].isna()
)

# 2️.Zero Values ​​(Dummy Data)
df["zero_geo_flag"] = (
    (df["Latitude"] == 0) | (df["Longitude"] == 0)
)

# 3.Incorrect longitude direction (California should be negative).
df["invalid_longitude_flag"] = (
    df["Longitude"] > 0
)

# 4️. Outside the California area (approximate range)
df["out_of_ca_flag"] = (
    (df["Latitude"] < 32) | (df["Latitude"] > 42) |
    (df["Longitude"] < -125) | (df["Longitude"] > -114)
)

# 5. Overall Objectives
df["invalid_geo_flag"] = (
    df["missing_geo_flag"] |
    df["zero_geo_flag"] |
    df["invalid_longitude_flag"] |
    df["out_of_ca_flag"]
)

# Convert to 0 / 1
geo_cols = [
    "missing_geo_flag",
    "zero_geo_flag",
    "invalid_longitude_flag",
    "out_of_ca_flag",
    "invalid_geo_flag"
]

for col in geo_cols:
    df[col] = df[col].astype(int)
print("Geographic Issues Count:")
for col in geo_cols:
    print(f"{col}: {df[col].sum()}")

Geographic Issues Count:
missing_geo_flag: 15822
zero_geo_flag: 25
invalid_longitude_flag: 29
out_of_ca_flag: 84
invalid_geo_flag: 15906


There are 15,906 records with geographic issues. Most of them are missing latitude or longitude (15,822). A few records have zero values, invalid longitude, or are outside California. Missing coordinates are the main problem.